# Fundamentos de NumPy
Criação e operações com arrays, desempenho em relação às listas, broadcasting, leitura de dados com `np.loadtxt`, fatiamento, tratamento de valores ausentes, uma regressão linear simples calculada "à mão" e geração de números aleatórios, usando preços de maçãs em cidades da Rússia.

**Origem:** Alura — curso de NumPy (formação Python para Data Science).

**Dados:** `apples_ts.csv` (arquivo do curso) — envie para `/content/` no Colab antes de executar a seção 4. As seções 1 a 3 não dependem de arquivos.

**Índice**
1. Criando arrays
2. Desempenho: lista x array
3. Broadcasting
4. Carregando e inspecionando dados
5. Fatiamento e comparação de arrays
6. Valores ausentes e estatísticas
7. Regressão linear simples
8. Números aleatórios e escolha do coeficiente angular
9. Exercício: gráfico de um círculo

In [ ]:
# Setup
import time

import matplotlib.pyplot as plt
import numpy as np

## 1. Criando arrays
`np.arange(inicio, fim, passo)` gera uma sequência (o `fim` não é incluído) e `np.array()` converte uma lista em array. Dimensões: **escalar** (valor simples), **vetor** (sequência de valores, ex.: `(70, 50, 30, 10)`), **matriz 2D** (linhas e colunas) e arrays **3D** ou mais (mais dimensões).

In [ ]:
# Anos de Copa do Mundo de 2002 a 2102
ano_inicial = 2002
ano_final = 2102
anos_copa = np.arange(ano_inicial, ano_final + 1, 4)
anos_copa

In [ ]:
lista = [1, 2, 3, 4, 5]
array = np.array(lista)
print(lista)
print(array)

> Atribuir um array a outra variável (`b = a`) não cria uma cópia: as duas apontam para os mesmos dados. Use `np.copy(a)` para desvincular o novo array do original.

## 2. Desempenho: lista x array
Operações vetorizadas do NumPy são muito mais rápidas que laços sobre listas.

In [ ]:
# Lista e array com 1.000.000 de elementos
lista = list(range(1000000))
array = np.array(lista)

# Elevar ao quadrado cada elemento da lista
start_time = time.time()
lista_quadrado = [i ** 2 for i in lista]
list_time = time.time() - start_time

# Elevar ao quadrado cada elemento do array
start_time = time.time()
array_quadrado = array ** 2
array_time = time.time() - start_time

print('Tempo da operação com a lista: ', list_time)
print('Tempo da operação com o array: ', array_time)

## 3. Broadcasting
Em operações entre arrays de formatos diferentes, o NumPy "estende" o menor quando as dimensões são compatíveis: aqui o vetor `b` é somado a cada linha da matriz `a`.

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([10, 20])
a + b

## 4. Carregando e inspecionando dados
O arquivo tem uma linha por série (a primeira com as datas e as seguintes com os preços de cada cidade) e uma coluna por mês. `usecols` ignora a primeira coluna (nomes) e lê as 87 colunas mensais.

In [ ]:
df = np.loadtxt('/content/apples_ts.csv', delimiter=',', skiprows=0, usecols=np.arange(1, 88, 1))

print(df.ndim)    # número de dimensões
print(df.size)    # número de elementos
print(df.shape)   # (linhas, colunas)

In [ ]:
# Transpondo: cada linha passa a ser um mês e cada coluna uma série
df_transposta = df.T
df_transposta.shape

In [ ]:
precos = df_transposta[:, 1:6]

# A coluna 0 tem as datas em formato de texto numérico; para os gráficos usamos 1, 2, ..., 87
datas = np.arange(1, 88)

plt.plot(datas, precos[:, 0])
plt.show()

## 5. Fatiamento e comparação de arrays
Separamos os preços por cidade e o primeiro ano de Moscou em blocos de 12 meses. `np.array_equal` verifica se dois arrays são idênticos e `np.allclose` se são próximos dentro de uma tolerância.

In [ ]:
Moscow = precos[:, 0]
Kaliningrad = precos[:, 1]
Petersburg = precos[:, 2]
Krasnodar = precos[:, 3]
Ekaterinburg = precos[:, 4]

Moscow_ano1 = Moscow[0:12]
Moscow_ano2 = Moscow[12:24]
Moscow_ano3 = Moscow[24:36]
Moscow_ano4 = Moscow[36:48]

meses = np.arange(1, 13, 1)
plt.plot(meses, Moscow_ano1)
plt.plot(meses, Moscow_ano2)
plt.plot(meses, Moscow_ano3)
plt.plot(meses, Moscow_ano4)
plt.legend(['ano1', 'ano2', 'ano3', 'ano4'])
plt.show()

In [ ]:
print(np.array_equal(Moscow_ano1, Moscow_ano2))     # arrays idênticos?
print(np.array_equal(Moscow_ano3, Moscow_ano4))
print(np.allclose(Moscow_ano3, Moscow_ano4, 0.2))   # arrays próximos (tolerância relativa de 0,2)?

## 6. Valores ausentes e estatísticas
A série de Kaliningrado tem um valor ausente (`nan`) na posição 4. Contamos os `nan` com `np.isnan` e preenchemos com a média dos meses vizinhos.

In [ ]:
print(np.isnan(Kaliningrad).sum())   # quantidade de valores ausentes

Kaliningrad[4] = np.mean([Kaliningrad[3], Kaliningrad[5]])
print(np.isnan(Kaliningrad).sum())

plt.plot(datas, Kaliningrad)
plt.show()

In [ ]:
print(np.mean(Moscow))
print(np.mean(Kaliningrad))

## 7. Regressão linear simples
Ajustamos uma reta $y = aX + b$ aos preços de Moscou pelo método dos mínimos quadrados:

$$a = \frac{n\sum XY - \sum X \sum Y}{n\sum X^2 - (\sum X)^2} \qquad b = \bar{Y} - a\bar{X}$$

A norma `np.linalg.norm(Y - y)` mede a distância entre os dados e a reta (quanto menor, melhor o ajuste).

In [ ]:
Y = Moscow
X = datas
n = np.size(Moscow)

# Coeficiente angular (a)
a = (n * np.sum(X * Y) - np.sum(X) * np.sum(Y)) / (n * np.sum(X ** 2) - np.sum(X) ** 2)
print(a)

# Coeficiente linear (b)
b = np.mean(Y) - a * np.mean(X)
print(b)

y = a * X + b
np.linalg.norm(Moscow - y)

In [ ]:
# Dados, reta ajustada e duas previsões (meses 41,5 e 100)
plt.plot(datas, Moscow)
plt.plot(X, y)
plt.plot(41.5, 41.5 * a + b, '*r')
plt.plot(100, 100 * a + b, '*r')
plt.show()

## 8. Números aleatórios e escolha do coeficiente angular
`np.random` gera números aleatórios; `np.random.seed` fixa a semente para tornar o resultado reproduzível. Testamos 100 coeficientes angulares aleatórios (mantendo o `b` da regressão) e calculamos a norma de cada reta.

In [ ]:
np.random.randint(low=40, high=100, size=100)

In [ ]:
np.random.seed(16)
coef_angulares = np.random.uniform(low=0.10, high=0.90, size=100)

norma2 = np.array([])
for i in range(100):
    norma2 = np.append(norma2, np.linalg.norm(Moscow - (coef_angulares[i] * X + b)))

# Coeficiente com a menor norma (melhor ajuste entre os sorteados)
melhor = np.argmin(norma2)
coef_angulares[melhor], norma2[melhor]

In [ ]:
# Junta normas e coeficientes em duas colunas e salva em CSV
dados = np.column_stack([norma2, coef_angulares])
np.savetxt('dados.csv', dados, delimiter=',')
dados[:5]

## 9. Exercício: gráfico de um círculo
A circunferência de raio 1 é $x^2 + y^2 = 1$, ou seja, $y = \pm\sqrt{1 - x^2}$ para $x \in [-1, 1]$.

In [ ]:
x = np.arange(-1, 1.01, 0.0001)

y1 = np.sqrt(1 - x ** 2)
y2 = -np.sqrt(1 - x ** 2)

plt.plot(x, y1, 'r')
plt.plot(x, y2, 'r')
plt.title('Círculo')
plt.xlabel('Eixo x')
plt.ylabel('Eixo y')
plt.show()